In [1]:
import torch
import numpy as np
# from sklearn.cluster import KMeans
# from scipy.spatial import distance
from tqdm import tqdm
import pytraj as pt
from BAT import *
from GetTorsionList import *
from Dihedral_Transformer import ICONTransformer
from PredictDihedralTransformer import reconstruct_full_bat

In [2]:
device = 'cuda'

out_path = "./out/"
out_path_params = out_path + "Driver_XYZ_params/"
params_file_name = 'Driver_XYZ_Params'
top_path = "../traj/cyclicpeptide/TVGGVG/protein.prmtop"
path = "../traj/cyclicpeptide/TVGGVG/TVGGVG_all_rmrp_min.dcd"
pdb_name = "Driver_Cyclic_Peptide_Interpolation"

par_id = 200

# Load the latent space representation
all_z = np.load(out_path + 'visual/' + 'Driver_Cyclic_Peptide_Predictionlatent.npy')

# load trajectory
traj = pt.load(path, top_path)
xyz = torch.tensor(traj.xyz, dtype=torch.float32, device=device)

In [3]:
interpolated_points = np.load(out_path + 'visual/' + 'interpolated_latent_points_slerp.npy')

In [4]:
checkpoint = torch.load(out_path_params + str(par_id) + params_file_name, map_location=device)

/tmp/ipykernel_1833/1511310169.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(out_path_params + str(par_id) + params_file_name, map_location=dev

In [5]:
# Load torsion information
traj = pt.load(path, top_path)
torsion_XYZ_inds, n_atoms, root = getTorsionList(path, top_path,
                                                 multifragment=False)
root_XYZ_inds = [root[0].index, root[1].index, root[2].index, root[2].index, root[2].index]
prior_atoms = [sorted([a1, a2]) for (a0, a1, a2, a3) in torsion_XYZ_inds]
primary_torsion_indices = [prior_atoms.index(prior_atoms[n]) for n in range(len(prior_atoms))]

# Load model parameters
primary_torsions = checkpoint['primary_torsions']
angle_differences = checkpoint['angle_differences']
primary_indices = checkpoint['primary_indices']
all_bonds = checkpoint['all_bonds'].to(device)
all_angles = checkpoint['all_angles'].to(device)
non_primary_bonds = torch.tensor(checkpoint['non_primary_bonds'], device=device)
non_primary_angles = torch.tensor(checkpoint['non_primary_angles'], device=device)
non_primary_indices = checkpoint['non_primary_indices']
fixed_tors_ind = checkpoint['fixed_torsions']
fixed_tors_ind = torch.tensor(fixed_tors_ind, device=device, dtype=torch.long)

n_torsions = len(primary_torsions) - len(fixed_tors_ind)
n_feats = 2 * n_torsions  # sin(torsion), cos(torsion)
n_tors = len(primary_torsions)

# get primary bonds and angles for reconstruction
n1, n2, va, vb = Coords2MainVecs(xyz, primary_torsions)
dat = BondAngleTorsion(n1, n2, va, vb)
bonds = dat[0][0, :]
angles = dat[1][0, :]
flexible_mask = torch.ones(n_tors, dtype=torch.bool, device=device)
flexible_mask[fixed_tors_ind] = False
fixed_torsions = dat[2][0, fixed_tors_ind]

# Initialize and load the transformer model
model = ICONTransformer(n_feats).to(device)
model.load_state_dict(checkpoint['model'])
model.eval()

# Reconstruct the interpolated points
all_coords = []
batch_size = 64

# Use the same bonds and angles for all frames
bonds = bonds.unsqueeze(0).expand(batch_size, -1)
angles = angles.unsqueeze(0).expand(batch_size, -1)
fixed_torsions = fixed_torsions.unsqueeze(0).expand(batch_size, -1)

for i in tqdm(range(0, len(interpolated_points), batch_size), desc="Reconstructing coordinates"):
    batch = torch.tensor(interpolated_points[i:i + batch_size], device=device, dtype=torch.float32)
    bonds = bonds[:len(batch)]
    angles = angles[:len(batch)]
    fixed_torsions = fixed_torsions[:len(batch)]

    with torch.no_grad():
        # Use the first frame coordinates for root-based reconstruction
        xyz = torch.as_tensor(traj.xyz[:1], device=device, dtype=torch.float32)
        root_based = GetRoot(xyz, root_XYZ_inds).expand(len(batch), -1)
        root_atoms_XYZ = [xyz[:, i, :].expand(len(batch), -1) for i in root_XYZ_inds]

        # Process through transformer
        transformer_output = model.decode(batch)

        # Prepare output for reconstruction
        flexible_sin_torsions = transformer_output[:, :, 0]
        flexible_cos_torsions = transformer_output[:, :, 1]

        sin_torsions = torch.zeros((len(batch), n_tors), device=device)
        cos_torsions = torch.zeros((len(batch), n_tors), device=device)

        sin_torsions[:, fixed_tors_ind] = torch.sin(fixed_torsions)
        cos_torsions[:, fixed_tors_ind] = torch.cos(fixed_torsions)
        sin_torsions[:, flexible_mask] = flexible_sin_torsions
        cos_torsions[:, flexible_mask] = flexible_cos_torsions

        out = torch.cat([bonds, angles, sin_torsions, cos_torsions], dim=-1)

        # Reconstruct full BAT representation
        full_bat = reconstruct_full_bat(out, angle_differences, all_bonds, all_angles,
                                        primary_indices, non_primary_bonds, non_primary_angles,
                                        non_primary_indices, torsion_XYZ_inds)

        n_total = len(all_bonds)
        bondP = full_bat[:, :n_total]
        angleP = full_bat[:, n_total:2 * n_total]
        sin_torsionP = full_bat[:, 2 * n_total:3 * n_total]
        cos_torsionP = full_bat[:, 3 * n_total:]
        torsionP = torch.atan2(sin_torsionP, cos_torsionP)

        # Combine all BAT coordinates
        bat = torch.cat([root_based, bondP, angleP, torsionP], dim=-1)

        # Convert BAT to Cartesian coordinates
        xyzbat = Bat2Coords(bat, root_XYZ_inds, torsion_XYZ_inds, primary_torsion_indices, root_atoms_XYZ)

        coords = xyzbat.detach().cpu().numpy()
        all_coords.append(coords)

all_coords = np.concatenate(all_coords, axis=0)

# Save the interpolated trajectory
traj_interpolated = pt.Trajectory(top=top_path)
traj_interpolated.xyz = all_coords
pt.write_traj(out_path + pdb_name + '.dcd', traj_interpolated, overwrite=True)

print(f"Cluster centers saved to: {out_path}visual/cluster_centers.npy")
print(f"Interpolated points saved to: {out_path}visual/interpolated_latent_points_slerp.npy")
print("Interpolation and reconstruction complete. Output saved to:", out_path + pdb_name + '.dcd')


Root atoms: [Atom(name=H, mass=1.008, index=1), Atom(name=N, mass=14.01, index=0), Atom(name=CA, mass=12.01, index=2)]
[[12, 2, 0, 1], [4, 2, 0, 1], [3, 2, 0, 1], [13, 12, 2, 0], [14, 12, 2, 0], [6, 4, 2, 0], [10, 4, 2, 0], [5, 4, 2, 0], [16, 14, 12, 2], [15, 14, 12, 2], [9, 6, 4, 2], [8, 6, 4, 2], [7, 6, 4, 2], [11, 10, 4, 2], [28, 16, 14, 12], [18, 16, 14, 12], [17, 16, 14, 12], [29, 28, 16, 14], [30, 28, 16, 14], [24, 18, 16, 14], [20, 18, 16, 14], [19, 18, 16, 14], [32, 30, 28, 16], [31, 30, 28, 16], [27, 24, 18, 16], [26, 24, 18, 16], [25, 24, 18, 16], [23, 20, 18, 16], [22, 20, 18, 16], [21, 20, 18, 16], [35, 32, 30, 28], [34, 32, 30, 28], [33, 32, 30, 28], [36, 35, 32, 30], [37, 35, 32, 30], [39, 37, 35, 32], [38, 37, 35, 32], [42, 39, 37, 35], [41, 39, 37, 35], [40, 39, 37, 35], [43, 42, 39, 37], [44, 42, 39, 37], [46, 44, 42, 39], [45, 44, 42, 39], [58, 46, 44, 42], [48, 46, 44, 42], [47, 46, 44, 42], [59, 58, 46, 44], [60, 58, 46, 44], [54, 48, 46, 44], [50, 48, 46, 44], [49,

/mnt/d/UCR/Lab2/ICoNv2_1/TVGGVG_all/BAT.py:25: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647378361/work/aten/src/ATen/native/Cross.cpp:62.)
  n1 = torch.cross(-va, vb)  # n1 is normal vector to -va, vb
/home/davidh/anaconda3/envs/torch/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Reconstructing coordinates:   1%|▍                                                                         | 1/157 [00:00<00:50,  3.09it/s]

bat2coords called
bat2coords called
bat2coords called


Reconstructing coordinates:   4%|███▎                                                                      | 7/157 [00:00<00:10, 13.64it/s]

bat2coords called
bat2coords called
bat2coords called
bat2coords called


...

Reconstructing coordinates:  97%|█████████████████████████████████████████████████████████████████████▋  | 152/157 [00:09<00:00, 18.27it/s]

bat2coords called
bat2coords called
bat2coords called
bat2coords called


Reconstructing coordinates: 100%|████████████████████████████████████████████████████████████████████████| 157/157 [00:09<00:00, 16.92it/s]

bat2coords called
bat2coords called
bat2coords called
bat2coords called


Cluster centers saved to: ./out/visual/cluster_centers.npy
Interpolated points saved to: ./out/visual/interpolated_latent_points_slerp.npy
Interpolation and reconstruction complete. Output saved to: ./out/Driver_Cyclic_Peptide_Interpolation.dcd
